In [1]:
import sys
import os

# Add the parent directory to Python's search path
sys.path.append(os.path.abspath('..'))
from quantum_qr.config import get_key
from quantum_qr.qrng import generate_quantum_random_bits

from quantum_qr.payload import (
    compute_tag,
    build_payload,
    encode_payload,
    decode_payload,
    tags_to_secret
)

In [2]:
# Shared key
key = get_key()

# Original data
data = "pay alice $10"

# Real quantum nonce
nonce = generate_quantum_random_bits(128)

print("Nonce:")
print(nonce)

# Compute integrity tag
tag = compute_tag(
    key,
    data,
    nonce,
    n_bits=8
)

print("\nTag:")
print(tag)

# Build payload
payload = build_payload(
    data=data,
    nonce=nonce,
    tag=tag
)

# Encode for QR storage
encoded = encode_payload(payload)

print("\nEncoded Payload:")
print(encoded)

# Decode again
decoded = decode_payload(encoded)

print("\nDecoded Payload:")
print(decoded)

# Recompute expected tag
tag_expected = compute_tag(
    key,
    decoded["data"],
    decoded["nonce"],
    n_bits=8
)

# XOR bridge
secret = tags_to_secret(
    decoded["tag"],
    tag_expected
)

print("\nSecret:")
print(secret)

Counts: {'00100101100010011101010000101101011111010001101011011011011100001111000001001001100101000010001011011100100110001010010010110000': 1}

128 Random Bits:
[0, 0, 0, 0, 1, 1, 0, 1, 0, 0, 1, 0, 0, 1, 0, 1, 0, 0, 0, 1, 1, 0, 0, 1, 0, 0, 1, 1, 1, 0, 1, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 1, 0, 0, 1, 1, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 1, 1, 1, 1, 0, 0, 0, 0, 1, 1, 1, 0, 1, 1, 0, 1, 1, 0, 1, 1, 0, 1, 0, 1, 1, 0, 0, 0, 1, 0, 1, 1, 1, 1, 1, 0, 1, 0, 1, 1, 0, 1, 0, 0, 0, 0, 1, 0, 1, 0, 1, 1, 1, 0, 0, 1, 0, 0, 0, 1, 1, 0, 1, 0, 0, 1, 0, 0]
Nonce:
[0, 0, 0, 0, 1, 1, 0, 1, 0, 0, 1, 0, 0, 1, 0, 1, 0, 0, 0, 1, 1, 0, 0, 1, 0, 0, 1, 1, 1, 0, 1, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 1, 0, 0, 1, 1, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 1, 1, 1, 1, 0, 0, 0, 0, 1, 1, 1, 0, 1, 1, 0, 1, 1, 0, 1, 1, 0, 1, 0, 1, 1, 0, 0, 0, 1, 0, 1, 1, 1, 1, 1, 0, 1, 0, 1, 1, 0, 1, 0, 0, 0, 0, 1, 0, 1, 0, 1, 1, 1, 0, 0, 1, 0, 0, 0, 1, 1, 0, 1, 0, 0, 1, 0, 0]

Tag:
11000101

Encoded Payload:
eyJ2ZXJzaW9uIjoiMSIsImRhdGEi

In [3]:
assert secret == "00000000"

print("✓ Authentic payload verified")

✓ Authentic payload verified


In [4]:
tampered_payload = decoded.copy()

tampered_payload["data"] = "pay bob $10000"

print("Tampered Payload:")
print(tampered_payload)

Tampered Payload:
{'version': '1', 'data': 'pay bob $10000', 'nonce': '00001101001001010001100100111011010001000010100110010010000011110000111011011011010110001011111010110100001010111001000110100100', 'tag': '11000101'}


In [5]:
tampered_expected_tag = compute_tag(
    key,
    tampered_payload["data"],
    tampered_payload["nonce"],
    n_bits=8
)

print("Expected Tag:")
print(tampered_expected_tag)

print("Observed Tag:")
print(tampered_payload["tag"])

Expected Tag:
00100000
Observed Tag:
11000101


In [6]:
tamper_secret = tags_to_secret(
    tampered_payload["tag"],
    tampered_expected_tag
)

print("Secret:")
print(tamper_secret)

Secret:
11100101


# Classical Tamper Detection Validation

Today I verified that the payload integrity mechanism works before introducing the quantum verification layer.

## Authentic Payload

For the original payload:

- data = "pay alice $10"
- nonce = quantum-generated
- tag = HMAC-derived integrity value

the verifier recomputed the expected tag and compared it against the observed tag.

The XOR result was:

00000000

This indicates that both tags match exactly.

---

## Tampered Payload

I modified the payload data:

"pay alice $10"
→
"pay bob $10000"

while keeping the original tag unchanged.

Since the attacker does not know the secret key, they cannot generate a valid replacement tag.

When the verifier recomputed the expected tag, the XOR result became non-zero.

This successfully detected tampering.

---

## Connection to the Quantum Layer

The XOR result:

secret = observed_tag XOR expected_tag

is exactly the secret string that will be passed into:

oracle_from_secret(secret)

in the next phase of the project.

Expected behavior:

- secret = 00000000
  → constant oracle
  → Deutsch–Jozsa measures all zeros
  → authentic

- secret ≠ 00000000
  → balanced oracle
  → Deutsch–Jozsa / Bernstein–Vazirani recovers the secret
  → tampered

This experiment confirms that the classical integrity pipeline works correctly before integrating the quantum verification circuit.